### Load and Store Data, also Postprocess Datasets

In [1]:
# %% [Setup — imports + paths]
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"]      = "1"
os.environ["MKL_NUM_THREADS"]      = "1"

import sys
from pathlib import Path
import xarray as xr
import dask
dask.config.set(scheduler="synchronous")

HELPER_DIR = Path("/nird/home/lbal/internship_storm_hans/helper")
if str(HELPER_DIR) not in sys.path:
    sys.path.insert(0, str(HELPER_DIR))

import config_paths as cfg
print("Setup done. Postprocessed root:", cfg.POSTPROC_DIR)

Setup done. Postprocessed root: /nird/datalake/NS9873K/lbal/postprocessed


### Catchment Weights

In [2]:
# %% [Catchment weights — build once; existing files are skipped]
# generate_weights.run_* read the grid from one raw file and skip any weight
# file that already exists, so this is safe to re-run.
# NOTE: era5 0.5x0.5 and seNorge weight files come from their own (unchanged)
# pipelines and are assumed to already exist on disk.
from generate_weights import run_era5_025, run_gfdl_spear, run_cesm2_le

run_era5_025()
run_gfdl_spear()
run_cesm2_le()
print("\nDone. Catchment weight files are up to date.")

────────────────────────────────────────────────────────────
ERA5 0.25x0.25 weight generation
────────────────────────────────────────────────────────────
Reading grid from ERA5 raw files ...
  Grid: 163 lats × 289 lons

[skip] Already exists: weights_catchment_nevina_bergheim_era5_0.25x0.25.nc
[skip] Already exists: weights_catchment_nevina_honnefoss_era5_0.25x0.25.nc
[skip] Already exists: weights_catchment_nevina_losna_era5_0.25x0.25.nc
[skip] Already exists: weights_catchment_regine_drammen_era5_0.25x0.25.nc
[skip] Already exists: weights_catchment_regine_glomma_era5_0.25x0.25.nc
[skip] Already exists: weights_catchment_regine_drammen_glomma_era5_0.25x0.25.nc
Done. Weight files written to:  /nird/datalake/NS9873K/lbal/postprocessed/weights
────────────────────────────────────────────────────────────
GFDL-SPEAR-MED-LE weight generation
────────────────────────────────────────────────────────────
Reading grid from: /nird/datalake/NS9873K/etdu/raw/smile/gfdl_spear_med_le/scandinavia/t

### Overall Precipitation Caches for ReturnPeriod Analysis

In [ ]:
# %% [Overall precipitation caches — run once; force=False skips existing]
from data_era5    import save_era5_overall
from data_senorge import save_senorge_overall
from data_smile   import save_smile_overall

FORCE_RECOMPUTE_OVERALL = False

print("ERA5 0.5° overall cache ...")
save_era5_overall("0.5x0.5", cfg.ERA5_RAW_DIR, cfg.overall_precip_path,
                  cfg.OVERALL_PRECIP_EXTENT, force=FORCE_RECOMPUTE_OVERALL)

print("\nERA5 0.25° overall cache ...")
save_era5_overall("0.25x0.25", cfg.ERA5_RAW_DIR, cfg.overall_precip_path,
                  cfg.OVERALL_PRECIP_EXTENT, force=FORCE_RECOMPUTE_OVERALL)

print("\nSeNorge overall cache ...")
save_senorge_overall(cfg.SENORGE_RAW_DIR, cfg.overall_precip_path,
                     cfg.OVERALL_PRECIP_EXTENT, force=FORCE_RECOMPUTE_OVERALL)

for _ds in cfg.SMILE_CONFIG:
    print(f"\n{_ds} overall cache ...")
    save_smile_overall(
        _ds,
        cfg.SMILE_CONFIG[_ds]["model_dir"],
        lambda mid, s, e, d=_ds: cfg.overall_precip_member_path(d, mid, s, e),
        force=FORCE_RECOMPUTE_OVERALL,
        unit_mode=cfg.SMILE_CONFIG[_ds].get("tp24_unit_mode", "auto"),
    )

print("\nDone. All overall 1-day precipitation caches are up to date.")


ERA5 0.5° overall cache ...
  Building ERA5 0.5x0.5 DAILY cache (1941–2024) ...


/nird/home/lbal/internship_storm_hans/helper/data_era5.py:190: UserWarning: The specified chunks separate the stored chunks along dimension "latitude" starting at index 64. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset(
/nird/home/lbal/internship_storm_hans/helper/data_era5.py:190: UserWarning: The specified chunks separate the stored chunks along dimension "longitude" starting at index 64. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset(
/nird/home/lbal/internship_storm_hans/helper/data_era5.py:190: UserWarning: The specified chunks separate the stored chunks along dimension "latitude" starting at index 64. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset(
/nird/home/lbal/internship_storm_hans/helper/data_era5.py:190: UserWarning: The specified chunks separate the stored chunks along dimension "longitude" starting at index 64. 

  [saved] post_processed_era5_0.5x0.5_1day_1941-2024.nc  ({'time': 30681, 'latitude': 22, 'longitude': 29})

ERA5 0.25° overall cache ...
  Building ERA5 0.25x0.25 DAILY cache (1941–2024) ...


/nird/home/lbal/internship_storm_hans/helper/data_era5.py:190: UserWarning: The specified chunks separate the stored chunks along dimension "latitude" starting at index 64. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset(
/nird/home/lbal/internship_storm_hans/helper/data_era5.py:190: UserWarning: The specified chunks separate the stored chunks along dimension "longitude" starting at index 64. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset(
/nird/home/lbal/internship_storm_hans/helper/data_era5.py:190: UserWarning: The specified chunks separate the stored chunks along dimension "latitude" starting at index 64. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset(
/nird/home/lbal/internship_storm_hans/helper/data_era5.py:190: UserWarning: The specified chunks separate the stored chunks along dimension "longitude" starting at index 64. 

  [saved] post_processed_era5_0.25x0.25_1day_1941-2024.nc  ({'time': 30681, 'latitude': 43, 'longitude': 57})

SeNorge overall cache ...
  Building SeNorge DAILY cache (1957–2025) ...


...

  [units] tp24 raw units='mm/day' -> mm (auto: already mm or kg m-2)
  [saved] post_processed_gfdl_spear_med_le_1day_member28_1921-2040.nc  ({'time': 43830, 'lat': 41, 'lon': 49})
    [build] member 29 (29/30) ...


/nird/home/lbal/internship_storm_hans/helper/data_smile.py:183: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset(
/nird/home/lbal/internship_storm_hans/helper/data_smile.py:183: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset(
/nird/home/lbal/internship_storm_hans/helper/data_smile.py:183: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset(
/nird/home/lbal/internship_storm_hans/helper/data_smile.py:183: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could

  [units] tp24 raw units='mm/day' -> mm (auto: already mm or kg m-2)
  [saved] post_processed_gfdl_spear_med_le_1day_member29_1921-2040.nc  ({'time': 43830, 'lat': 41, 'lon': 49})
    [build] member 30 (30/30) ...


/nird/home/lbal/internship_storm_hans/helper/data_smile.py:183: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset(
/nird/home/lbal/internship_storm_hans/helper/data_smile.py:183: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset(
/nird/home/lbal/internship_storm_hans/helper/data_smile.py:183: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset(
/nird/home/lbal/internship_storm_hans/helper/data_smile.py:183: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could

  [units] tp24 raw units='mm/day' -> mm (auto: already mm or kg m-2)
  [saved] post_processed_gfdl_spear_med_le_1day_member30_1921-2040.nc  ({'time': 43830, 'lat': 41, 'lon': 49})

Done. All overall 1-day precipitation caches are up to date.


### ERA5 Interpolated Precipitation Caches for Compound Extremes Analysis

In [ ]:
# %% [ERA5-interpolated overall precipitation cache — run once]
from data_era5 import save_era5_interpolated_overall

ERA5_INTERP_TP_DIR = cfg.ERA5_INTERPOLATED_BASE / "tp"

print("ERA5 interpolated overall precipitation cache ...")
save_era5_interpolated_overall(
    ERA5_INTERP_TP_DIR,
    cfg.overall_precip_path,
    cfg.OVERALL_PRECIP_EXTENT,
    force=False,
)
print("Done.")

ERA5 interpolated overall precipitation cache ...
  Building ERA5-interpolated DAILY cache (1941–2025) ...
  ERA5-interp 'tp': units='mm/day', conversion factor=1.0
  [saved] post_processed_era5_interpolated_1day_1941-2025.nc  ({'time': 31046, 'lat': 12, 'lon': 12})
Done.


### ERA5 Interpolated + CESM2-le SWE + Soil Moisture Caches for Compound Extremes Analysis

In [ ]:
# %% [SWE & soil-moisture daily caches — run once; set FORCE=True to rebuild]
# Stores the WHOLE available daily timeseries as postprocessed caches:
#   per-member CESM2-LE (1920–2034) and ERA5-interp (full record) under
#   cesm2_le/{swe,soil_moisture} and era5_interpolated/{swe,soil_moisture}.
from data_smile import save_cesm2_le_field_overall
from data_era5  import save_era5_interpolated_field_overall

FORCE_RECOMPUTE_FIELDS = False

# Minimal per-variable I/O config (kind = postprocessed sub-folder).
FIELD_VARIABLES = [
    dict(kind="swe",           cesm2_dir=cfg.CESM2_LE_SWE_DIR,
         era5_interp_dir=cfg.ERA5_INTERPOLATED_SWE_DIR,
         cesm2_raw_var="SWE", era5_raw_var="sd",   noun="Snowmelt (SWE)"),
    dict(kind="soil_moisture", cesm2_dir=cfg.CESM2_LE_SM_DIR,
         era5_interp_dir=cfg.ERA5_INTERPOLATED_SWVL_DIR,
         cesm2_raw_var="SM",  era5_raw_var="swvl", noun="Soil Moisture"),
]

for _V in FIELD_VARIABLES:
    k = _V["kind"]
    print(f"\n=== {_V['noun']} — per-member daily CESM2-LE caches ===")
    save_cesm2_le_field_overall(
        _V["cesm2_dir"],
        (lambda mid, s, e, k=k: cfg.field_daily_cache_path("cesm2_le", k, s, e, member_id=mid)),
        variable=_V["cesm2_raw_var"], cache_var=k, units="kg/m2",
        force=FORCE_RECOMPUTE_FIELDS)
    print(f"\n=== {_V['noun']} — ERA5-interpolated daily cache ===")
    save_era5_interpolated_field_overall(
        _V["era5_interp_dir"],
        (lambda s, e, k=k: cfg.field_daily_cache_path("era5_interpolated", k, s, e)),
        variable=_V["era5_raw_var"], cache_var=k, units="kg/m2",
        extent=cfg.OVERALL_PRECIP_EXTENT, force=FORCE_RECOMPUTE_FIELDS)

print("\nAll SWE & soil-moisture daily caches up to date.")


=== Snowmelt (SWE) — per-member daily CESM2-LE caches ===
  cesm2_le/swe: 90 members, 1920–2034 ...
    [build] member 002 (1/90) ...
  [time] Removing 1 duplicate timestamp(s) ...
  [saved] post_processed_cesm2_le_swe_1day_member002_1920-2034.nc  ({'time': 41975, 'lat': 22, 'lon': 25})
    [build] member 004 (2/90) ...
  [time] Removing 1 duplicate timestamp(s) ...
  [saved] post_processed_cesm2_le_swe_1day_member004_1920-2034.nc  ({'time': 41975, 'lat': 22, 'lon': 25})
    [build] member 006 (3/90) ...
  [time] Removing 1 duplicate timestamp(s) ...
  [saved] post_processed_cesm2_le_swe_1day_member006_1920-2034.nc  ({'time': 41975, 'lat': 22, 'lon': 25})
    [build] member 008 (4/90) ...
  [time] Removing 1 duplicate timestamp(s) ...
  [saved] post_processed_cesm2_le_swe_1day_member008_1920-2034.nc  ({'time': 41975, 'lat': 22, 'lon': 25})
    [build] member 010 (5/90) ...
  [time] Removing 1 duplicate timestamp(s) ...
  [saved] post_processed_cesm2_le_swe_1day_member010_1920-2034.nc 

### Snowmelt Calculation and Storage

In [ ]:
# %% [N-day ΔSWE snowmelt cache builder — run once per window; FORCE=True to rebuild]# %% [N-day snowmelt cache builder — run once per window; FORCE=True to rebuild]
# Builds the daily snowmelt caches consumed by compound_flood_risk_analysis.ipynb.
# For each date t the stored value is a POSITIVE snowmelt magnitude
#     max(0, −(SWE(t) − SWE(t−(WINDOW_DAYS_SWE−1))))
# i.e. the SWE *decrease* over a WINDOW_DAYS_SWE-day window (bigger SWE drop →
# bigger positive snowmelt); any SWE *gain* maps to 0. So the field holds zeros +
# positives (the sign is flipped vs. the raw ΔSWE), which keeps the 90th-pctl.
# analysis meaningful (high-melt days sit at the top). Positive-only step done with
# rolling_melt = (−ΔSWE).clip(min=0.0), NOT np.ceil. Computed over the WHOLE
# available record (CESM2-LE 1920–2034, ERA5-interp full record) and saved next to
# the raw 1-day caches with the window label in the filename (…_swe_{N}day_…).
# Select WINDOW_DAYS_SWE ∈ {2, 3, 4}; re-run once per window you want available.
from data_smile      import save_cesm2_le_field_diff_overall
from data_era5       import save_era5_interpolated_field_diff_overall
from catchment_tools import rolling_melt, open_field_cache

WINDOW_DAYS_SWE      = 2       # ← window N for the snowmelt difference (2 / 3 / 4)
FORCE_RECOMPUTE_DSWE = False    # rebuild: existing caches hold old negative values

if WINDOW_DAYS_SWE < 2:
    raise ValueError("WINDOW_DAYS_SWE must be >= 2 (ΔSWE = SWE(t) − SWE(t−(N−1))).")

_open_swe = lambda p, s, e: open_field_cache(p, "swe", s, e)

print(f"=== {WINDOW_DAYS_SWE}-day snowmelt — per-member CESM2-LE caches ===")
save_cesm2_le_field_diff_overall(
    cfg.CESM2_LE_SWE_DIR,
    in_path_fn  = (lambda mid, s, e: cfg.field_daily_cache_path("cesm2_le", "swe", s, e, member_id=mid)),
    out_path_fn = (lambda mid, s, e: cfg.field_daily_cache_path("cesm2_le", "swe", s, e, member_id=mid, window_days=WINDOW_DAYS_SWE)),
    cache_var="swe", diff_fn=rolling_melt, open_cache_fn=_open_swe,
    window_days=WINDOW_DAYS_SWE, units="kg/m2", force=FORCE_RECOMPUTE_DSWE)

print(f"\n=== {WINDOW_DAYS_SWE}-day snowmelt — ERA5-interpolated cache ===")
save_era5_interpolated_field_diff_overall(
    cfg.ERA5_INTERPOLATED_SWE_DIR,
    in_path_fn  = (lambda s, e: cfg.field_daily_cache_path("era5_interpolated", "swe", s, e)),
    out_path_fn = (lambda s, e: cfg.field_daily_cache_path("era5_interpolated", "swe", s, e, window_days=WINDOW_DAYS_SWE)),
    cache_var="swe", diff_fn=rolling_melt, open_cache_fn=_open_swe,
    window_days=WINDOW_DAYS_SWE, units="kg/m2", force=FORCE_RECOMPUTE_DSWE)

print(f"\n{WINDOW_DAYS_SWE}-day snowmelt caches up to date.")


=== 2-day snowmelt — per-member CESM2-LE caches ===
  cesm2_le/swe: 90 members, 2-day ΔSWE, 1920–2034 ...
    [build] member 002 (1/90) ...


  [saved] post_processed_cesm2_le_swe_2day_member002_1920-2034.nc  ({'time': 41975, 'lat': 22, 'lon': 25})
    [build] member 004 (2/90) ...
  [saved] post_processed_cesm2_le_swe_2day_member004_1920-2034.nc  ({'time': 41975, 'lat': 22, 'lon': 25})
    [build] member 006 (3/90) ...
  [saved] post_processed_cesm2_le_swe_2day_member006_1920-2034.nc  ({'time': 41975, 'lat': 22, 'lon': 25})
    [build] member 008 (4/90) ...
  [saved] post_processed_cesm2_le_swe_2day_member008_1920-2034.nc  ({'time': 41975, 'lat': 22, 'lon': 25})
    [build] member 010 (5/90) ...
  [saved] post_processed_cesm2_le_swe_2day_member010_1920-2034.nc  ({'time': 41975, 'lat': 22, 'lon': 25})
    [build] member 012 (6/90) ...
  [saved] post_processed_cesm2_le_swe_2day_member012_1920-2034.nc  ({'time': 41975, 'lat': 22, 'lon': 25})
    [build] member 014 (7/90) ...
  [saved] post_processed_cesm2_le_swe_2day_member014_1920-2034.nc  ({'time': 41975, 'lat': 22, 'lon': 25})
    [build] member 016 (8/90) ...
  [saved] pos

### CESM2-LE Catchment-Averaged Compound Series (Precipitation Sum, Soil-Moisture Mean, Snowmelt)


In [2]:
# %% CESM2-LE compound extremes analysis:
# Per default full record 1920–2034
# only members common to ALL three variables (90 — SM/SWE lack odd members 001–019) considered
# One .nc per window, catchment, variable, dims [member, time], saved to postprocessed/cesm2_le/catchment_averaged/

from catchment_tools import save_cesm2_le_catchment_field_series

# ── Selection ─────────────────────────────────────────────────────────────────
WINDOW_DAYS_COMPOUND     = 2                              # rolling window: 2 / 3 / 4 / … (snowmelt needs ≥ 2)
COMPOUND_SLUGS           = list(cfg.COMPOUND_CATCHMENTS)  # regine_drammen, regine_glomma, regine_drammen_glomma
COMPOUND_VARIABLES       = ["precipitation", "soil_moisture", "snowmelt"]
FORCE_RECOMPUTE_COMPOUND = False

for _slug in COMPOUND_SLUGS:
    for _var in COMPOUND_VARIABLES:
        print(f"\n=== {_var} — {_slug} — {WINDOW_DAYS_COMPOUND}-day ===")
        save_cesm2_le_catchment_field_series(
            variable=_var, window_days=WINDOW_DAYS_COMPOUND,
            catchment_slug=_slug, force=FORCE_RECOMPUTE_COMPOUND)

print("\nAll CESM2-LE catchment compound series up to date.")


=== precipitation — regine_drammen — 2-day ===
  [skip] exists: post_processed_cesm2_le_2day_regine_drammen_precipitation_1920-2034.nc

=== soil_moisture — regine_drammen — 2-day ===
  [skip] exists: post_processed_cesm2_le_2day_regine_drammen_soil_moisture_1920-2034.nc

=== snowmelt — regine_drammen — 2-day ===
  [skip] exists: post_processed_cesm2_le_2day_regine_drammen_snowmelt_1920-2034.nc

=== precipitation — regine_glomma — 2-day ===
  [skip] exists: post_processed_cesm2_le_2day_regine_glomma_precipitation_1920-2034.nc

=== soil_moisture — regine_glomma — 2-day ===
  [skip] exists: post_processed_cesm2_le_2day_regine_glomma_soil_moisture_1920-2034.nc

=== snowmelt — regine_glomma — 2-day ===
  [skip] exists: post_processed_cesm2_le_2day_regine_glomma_snowmelt_1920-2034.nc

=== precipitation — regine_drammen_glomma — 2-day ===
  [skip] exists: post_processed_cesm2_le_2day_regine_drammen_glomma_precipitation_1920-2034.nc

=== soil_moisture — regine_drammen_glomma — 2-day ===
  [sk